# 🎬 Reco-Spark ALS Model Training Pipeline

PySpark ALS model training on the **MovieLens 25M** dataset.

| Step | Description |
|------|----------|
| 1 | Environment Setup (PySpark + Java) |
| 2 | Dataset Download & Loading |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Data Cleaning |
| 5 | Train/Test Split (80% / 20%) |
| 6 | ALS + CrossValidator Hyperparameter Tuning |
| 7 | Model Evaluation (RMSE < 1.0) |
| 8 | Model Export & Download |

---
## 📦 Step 1: Environment Setup

In [ ]:
# Install PySpark
!pip install pyspark==3.5.4 -q

In [ ]:
# Check Java (Usually pre-installed in Colab)
!java -version

In [ ]:
import os
import time
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType, StringType
from pyspark.ml.recommendation import ALS, ALSModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("All imports successful! ✅")

---
## 📥 Step 2: Dataset Download & Loading

In [ ]:
# Download MovieLens 25M dataset
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
DATA_DIR = "/content/data"
EXTRACT_DIR = f"{DATA_DIR}/ml-25m"

if not os.path.exists(f"{EXTRACT_DIR}/ratings.csv"):
    print("📥 Downloading MovieLens 25M... (~250MB)")
    !mkdir -p {DATA_DIR}
    !wget -q --show-progress -O {DATA_DIR}/ml-25m.zip {DATASET_URL}
    print("📦 Extracting archive...")
    !unzip -q -o {DATA_DIR}/ml-25m.zip -d {DATA_DIR}
    !rm {DATA_DIR}/ml-25m.zip
    print("✅ Dataset is ready!")
else:
    print("✅ Dataset already exists!")

# File sizes
for f in os.listdir(EXTRACT_DIR):
    path = os.path.join(EXTRACT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"  📄 {f}: {size_mb:.1f} MB")

In [ ]:
RATINGS_PATH = f"{EXTRACT_DIR}/ratings.csv"
MOVIES_PATH = f"{EXTRACT_DIR}/movies.csv"
MODEL_SAVE_DIR = "/content/als_model"

# Create SparkSession
spark = (
    SparkSession.builder
    .appName("Reco-Spark-ALS-Training")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"✅ SparkSession is ready! (version: {spark.version})")

---
## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Define Schema
ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", FloatType(), False),
    StructField("timestamp", LongType(), True),
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField("genres", StringType(), True),
])

# Load data
print("📂 Loading Ratings...")
ratings = spark.read.option("header", "true").schema(ratings_schema).csv(RATINGS_PATH)

print("📂 Loading Movies...")
movies = spark.read.option("header", "true").schema(movies_schema).csv(MOVIES_PATH)

ratings.cache()
movies.cache()

print("✅ Data loaded successfully!")

In [ ]:
# Schema info
print("📋 Ratings Schema:")
ratings.printSchema()
print("📋 Movies Schema:")
movies.printSchema()

In [ ]:
# Size and statistics
rating_count = ratings.count()
user_count = ratings.select("userId").distinct().count()
unique_movies = ratings.select("movieId").distinct().count()
movie_count = movies.count()

print(f"📊 Total ratings        : {rating_count:,}")
print(f"📊 Unique users         : {user_count:,}")
print(f"📊 Unique movies        : {unique_movies:,}")
print(f"📊 Total movie rows     : {movie_count:,}")

In [ ]:
# Rating distribution
print("📊 Rating Distribution:")
ratings.groupBy("rating").count().orderBy("rating").show()

print("📊 Rating Statistics:")
ratings.select("rating").describe().show()

In [ ]:
# Sample data
print("📊 Sample Ratings:")
ratings.show(5, truncate=False)

print("📊 Sample Movies:")
movies.show(5, truncate=False)

---
## 🧹 Step 4: Data Cleaning

In [ ]:
before = ratings.count()
ratings_clean = ratings.dropna(subset=["userId", "movieId", "rating"])
after = ratings_clean.count()

print(f"  Rows before cleaning : {before:,}")
print(f"  Rows after cleaning  : {after:,}")
print(f"  Rows dropped         : {before - after:,}")

# Keep only necessary columns for ALS
ratings_clean = ratings_clean.select("userId", "movieId", "rating")
print("\n✅ Data cleaned!")

---
## ✂️ Step 5: Train/Test Split (80% / 20%)

In [ ]:
train, test = ratings_clean.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()

train_count = train.count()
test_count = test.count()
total = train_count + test_count

print(f"  Train set: {train_count:,} rows ({train_count/total*100:.1f}%)")
print(f"  Test set : {test_count:,} rows ({test_count/total*100:.1f}%)")
print("\n✅ Train/Test split completed!")

---
## 🤖 Step 6: ALS + CrossValidator Hyperparameter Tuning

**Grid:**
- `rank`: [10, 50]
- `regParam`: [0.01, 0.1]
- `maxIter`: [5, 10]

Total: **8 combinations × 3 folds = 24 models**

In [ ]:
# ALS definition
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    coldStartStrategy="drop",
)

# Hyperparameter grid
param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [10, 50])
    .addGrid(als.regParam, [0.01, 0.1])
    .addGrid(als.maxIter, [5, 10])
    .build()
)

# Evaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction",
)

# CrossValidator
cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=1,
    seed=42,
)

print(f"Grid size: {len(param_grid)} combinations")
print(f"Total models to train (3-fold CV): {len(param_grid) * 3}")
print("\n⏳ Starting CrossValidator... (This might take 15-30 minutes)")

start_time = time.time()
cv_model = cv.fit(train)
elapsed = time.time() - start_time

print(f"\n✅ CrossValidator completed! Elapsed time: {elapsed/60:.1f} minutes")

---
## 📈 Step 7: Model Evaluation

In [ ]:
best_model = cv_model.bestModel
predictions = best_model.transform(test)
rmse = evaluator.evaluate(predictions)

print(f"📈 Test RMSE: {rmse:.4f}")
print(f"🎯 Target:    RMSE < 1.0")
print(f"{'\u2705 TARGET ACHIEVED!' if rmse < 1.0 else '\u274c TARGET FAILED'}")

# Best parameters
print(f"\n🏆 Best Parameters:")
print(f"  rank     = {best_model.rank}")
print(f"  maxIter  = {best_model._java_obj.parent().getMaxIter()}")
print(f"  regParam = {best_model._java_obj.parent().getRegParam()}")

In [ ]:
# Cross Validation results - all combinations
avg_metrics = cv_model.avgMetrics

print("📊 Cross Validation Results (RMSE):")
print("-" * 50)

results = []
for i, (params, metric) in enumerate(zip(param_grid, avg_metrics)):
    rank_val = params[als.rank]
    reg_val = params[als.regParam]
    iter_val = params[als.maxIter]
    results.append((i+1, rank_val, reg_val, iter_val, metric))
    print(f"  #{i+1:2d}: rank={rank_val:3d}, regParam={reg_val:.2f}, maxIter={iter_val:2d} → RMSE={metric:.4f}")

# Best combination
best_idx = avg_metrics.index(min(avg_metrics))
print(f"\n🏆 Best: #{best_idx+1} (RMSE = {min(avg_metrics):.4f})")

In [ ]:
# Sample recommendations for User 1
print("🎬 Top-10 Movie Recommendations for User 1:")
print("-" * 60)

user_recs = best_model.recommendForAllUsers(10)
user1_recs = user_recs.filter(user_recs.userId == 1).collect()

if user1_recs:
    for j, rec in enumerate(user1_recs[0].recommendations, 1):
        movie_row = movies.filter(movies.movieId == rec.movieId).collect()
        title = movie_row[0].title if movie_row else "Unknown"
        genres = movie_row[0].genres if movie_row else "-"
        print(f"  {j:2d}. {title}")
        print(f"      Genre: {genres} | Predicted Rating: {rec.rating:.2f}")

---
## 💾 Step 8: Model Export & Download

In [ ]:
# Save model
if os.path.exists(MODEL_SAVE_DIR):
    shutil.rmtree(MODEL_SAVE_DIR)

best_model.write().overwrite().save(MODEL_SAVE_DIR)
print(f"💾 Model saved to: {MODEL_SAVE_DIR}")

# Save metadata
metadata_path = "/content/model_metadata.txt"
with open(metadata_path, "w") as f:
    f.write(f"Model: ALS (PySpark 3.5.4)\n")
    f.write(f"Dataset: MovieLens 25M\n")
    f.write(f"Test RMSE: {rmse:.4f}\n")
    f.write(f"Rank: {best_model.rank}\n")
    f.write(f"MaxIter: {best_model._java_obj.parent().getMaxIter()}\n")
    f.write(f"RegParam: {best_model._java_obj.parent().getRegParam()}\n")
    f.write(f"Training Time: {elapsed/60:.1f} minutes\n")
    f.write(f"Trained at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"\nAll CV Results (RMSE):\n")
    for i, (params, metric) in enumerate(zip(param_grid, avg_metrics)):
        f.write(f"  #{i+1}: rank={params[als.rank]}, regParam={params[als.regParam]}, maxIter={params[als.maxIter]} -> RMSE={metric:.4f}\n")

print(f"📝 Metadata saved to: {metadata_path}")

In [ ]:
# Zip the model for downloading
ZIP_PATH = "/content/als_model_export"

!cp /content/model_metadata.txt {MODEL_SAVE_DIR}/
shutil.make_archive(ZIP_PATH, 'zip', MODEL_SAVE_DIR)

zip_size = os.path.getsize(f"{ZIP_PATH}.zip") / (1024*1024)
print(f"📦 Model archive: {ZIP_PATH}.zip ({zip_size:.1f} MB)")
print("\n⬇️ Run the cell below to download the model!")

In [ ]:
# Download model
from google.colab import files

print("⬇️ Starting model download...")
files.download(f"{ZIP_PATH}.zip")
files.download(metadata_path)
print("✅ Download completed!")

In [ ]:
# Validation: Reload model and test
print("🔄 Model validation - reloading...")
loaded_model = ALSModel.load(MODEL_SAVE_DIR)
loaded_predictions = loaded_model.transform(test)
loaded_rmse = evaluator.evaluate(loaded_predictions)
print(f"📈 Loaded model RMSE: {loaded_rmse:.4f}")
print(f"📈 Original RMSE:     {rmse:.4f}")
print(f"{'\u2705 Model validated!' if abs(loaded_rmse - rmse) < 0.0001 else '\u26a0\ufe0f RMSE difference found!'}")

In [ ]:
# Stop Spark session
spark.stop()
print("\n🔌 Spark session closed.")
print("\n" + "=" * 60)
print("  🎉 Pipeline completed successfully!")
print("=" * 60)
print(f"\n  📈 Final RMSE: {rmse:.4f}")
print(f"  🏆 Best Rank: {best_model.rank}")
print(f"  📦 Model file: als_model_export.zip")
print(f"\n  Next step: Download the ZIP file and")
print(f"  extract it to the ml_pipeline/models/als_model/ folder.")